# Part 1.5 Tracing

After this notebook you can open a claim's trace in LangSmith, label runs with tags and
metadata, and find them again with a filter.

In [ ]:
from kea.config import setup

setup("kea-claims-local")

## See it

Every run of the finished workflow is already traced. Each node is a span, each model
call is nested inside it, and the photo is attached to the root run.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from kea.claims.graph import builder

store = InMemoryStore()
graph = builder.compile(checkpointer=InMemorySaver(), store=store)


def claim(photo: str, passenger_id: str, flight_number: str, declared_value_usd: float) -> dict:
    return {
        "photo": photo,  # a file name under data/bag-photos, the same input Studio takes
        "passenger_id": passenger_id,
        "flight_number": flight_number,
        "declared_value_usd": declared_value_usd,
    }

In [ ]:
from langchain_core.tracers.context import collect_runs

with collect_runs() as collected:
    traced = graph.invoke(
        claim("wheel-01.jpg", "pax-1003", "KA203", 300),
        {"configurable": {"thread_id": "claim-1003"}},
    )

print(traced["status"], traced["payout_usd"])
print(collected.traced_runs[0].get_url())

Open the link.

<img src="../../assets/screenshots/p1-05-trace.png" alt="the trace of a claim in the kea-claims-local project: the node waterfall on the left, the photo attachment on the root run, and the validatephoto structured-output call expanded." width="900">

## Build it

`setup()` already turned tracing on with two environment variables. Check them, then
label a run with a name, tags, and metadata so it is easy to find later.

In [ ]:
import os

print("LANGSMITH_TRACING =", os.environ["LANGSMITH_TRACING"])
print("LANGSMITH_PROJECT =", os.environ["LANGSMITH_PROJECT"])

graph.invoke(
    claim("handle-02.jpg", "pax-1004", "KA204", 250),
    {
        "configurable": {"thread_id": "claim-1004"},
        "run_name": "claim handle-02",
        "tags": ["workshop", "handle"],
        "metadata": {"passenger_id": "pax-1004", "photo": "handle-02"},
    },
)

In the `kea-claims-local` project, click **Filter**, choose **Tag**, and pick
`handle`. Only the labeled run is left, named `claim handle-02`.

<img src="../../assets/screenshots/p1-05-filter.png" alt="the project's run list filtered to the handle tag, with the claim handle-02 row and its metadata visible." width="900">

## Under the hood

- The graph run is the **root run**. Nodes and model calls are child runs, so latency,
  tokens, and errors roll up the tree.
- A **project** groups traces. `setup("kea-claims-local")` sets
  `LANGSMITH_PROJECT`; everything this part runs lands there.
- **Tags** and **metadata** go in the run config. Filter on them in the UI, or
  query them with the LangSmith SDK.
- The photo is added to the root run in `attach_image_to_root_run`
  (`kea/claims/images.py`). That is why it shows in the UI and travels with a trace
  into a dataset.
- Code outside LangChain joins the same trace with the `@traceable` decorator from
  `langsmith`.
- `langgraph dev` and Studio runs trace too, to the project named in `.env`.

Docs: [Observability concepts](https://docs.langchain.com/langsmith/observability-concepts).

## Recap

- Tracing is on after `setup()`: one root run per claim, nodes and model calls below it.
- Tags and metadata make runs findable.
- The photo rides on the root run, so a trace can become a test case.

**Next:** [Part 1.6 Checkpointers](06_checkpointers.ipynb)